# Distribution of a sum (convolution)

_Probability & Statistics_

**Add two independent random variables. Their densities blend together by sliding one across the other.**

You have two independent random variables, $X$ and $Y$. You care about their sum $Z = X + Y$.

     The sum can hit a value $z$ in many ways: $X = k$ and $Y = z - k$, for every possible $k$.

---

This notebook builds the sum-of-variables idea one step at a time: convolve two discrete PMFs, check them against simulation, then do the continuous case, and finally picture the result. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

### Step 1 — Sum two dice by convolving their PMFs

For independent variables, the PMF of the sum `Z = X + Y` is the **convolution** of the two input PMFs. Each die face is uniform with probability `1/6`, so `np.convolve(face, face)` slides one PMF across the other and gives the probability of every total. The result's index `0` corresponds to a sum of `2` (the smallest two dice can total), so the sums run `2..12`.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Discrete: sum of two fair dice via convolution of the face PMFs.
face = np.full(6, 1/6)
pmf = np.convolve(face, face)              # index 0 -> sum 2
sums = np.arange(2, 13)

### Step 2 — Check the dice PMF against simulation

To confirm the convolution is right, we roll two dice two million times and add them up (Monte-Carlo). For each possible total we compare the convolution probability to the empirical fraction of rolls hitting it — they should agree to a few decimals.

In [ ]:
# Monte-Carlo check.
roll = rng.integers(1, 7, (2_000_000, 2)).sum(axis=1)

for s, p in zip(sums, pmf):
    mc = np.mean(roll == s)
    print(f"P(sum={s:2d}) conv={p:.4f}  MC={mc:.4f}")

### Step 3 — The continuous case: two normals

Convolution works for continuous densities too. Adding two independent `N(0,1)` variables gives `N(0,2)` — variances add. We confirm this two ways: the sample variance of the simulated sum should be near `2`, and the SciPy `N(0, sqrt(2))` density at a point matches the theoretical formula.

In [ ]:
from scipy import stats

# Continuous: N(0,1) + N(0,1) = N(0,2). Compare density at a point.
z = rng.standard_normal(2_000_000) + rng.standard_normal(2_000_000)

print("Var(Z) MC =", round(z.var(), 3), " theory = 2")
print("pdf@1 N(0,2) =", round(stats.norm(0, np.sqrt(2)).pdf(1.0), 4))

## Visualize the data & results

_What PMF do you get from convolving two flat distributions, and does it match simulation?_

### Step 1 — A tent from two short uniforms

Convolving two *flat* PMFs of different widths gives a **tent** (trapezoid-ish) shape. Here `A` is uniform on `{1,2,3}` and `B` is uniform on `{1,2}`. Their convolution covers sums `2..5`, and again index `0` maps to sum `2`.

In [ ]:
import numpy as np

# Chart 1: A on {1,2,3}, B on {1,2}, each uniform -> tent PMF over sums 2..5.
a = np.full(3, 1/3)                  # P(A) on 1..3
b = np.full(2, 1/2)                  # P(B) on 1..2
tent = np.convolve(a, b)            # index 0 -> sum 2
tent_sums = np.arange(2, 6)

### Step 2 — A triangle from two equal uniforms

When the two flat PMFs are the *same* width, the convolution is a symmetric **triangle**. Two fair dice (uniform on `1..6`) produce the familiar triangular PMF that peaks at `7`, the most likely total.

In [ ]:
# Chart 2: two fair dice, faces uniform on 1..6 -> triangle PMF peaking at 7.
face = np.full(6, 1/6)
dice = np.convolve(face, face)     # index 0 -> sum 2
dice_sums = np.arange(2, 13)

### Step 3 — Plot both PMFs side by side

Left is the tent, right is the triangle. Both come from the same `np.convolve` operation — the shape is determined entirely by the widths of the inputs you slide together.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar([str(s) for s in tent_sums], tent, color='#c89bff')
ax1.set_title('Tent PMF of Z = A + B (sums 2..5)')

ax2.bar([str(s) for s in dice_sums], dice, color='#4ea1ff')
ax2.set_title('Two fair dice: convolution PMF peaks at 7')

plt.tight_layout()
plt.show()